<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/07_TADP_Experiment_G_v20_1_Adversarial_Diagnostic_Probe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [3]:
# ======================================================================================
# TADP v20.1 — 2 scenarios - 3 seeds - TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GX and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# v20.0 governance policy:
#   - 28 factors across 6 dimensions:
#       dim1=4, dim2(DQ)=8, dim3=4, dim4=3, dim5=5, dim6=4.
#   - Documentary evidence is represented at the individual-factor level.
#   - DQ evidence is machine-measured from client TRAIN partitions only.
#   - HPS remains client-specific and uses the six frozen dimension weights.
#   - Mandatory Governance Gate is evaluated FIRST.
#       Missing/non-finite evidence or score below the policy minimum for any
#       mandatory factor -> immediate AUTO_REJECT. No compensation is permitted.
#   - Every averaged dimension must be >= 2.5/5 or the client is AUTO_REJECTED.
#   - HPS < 3.0 -> AUTO_REJECT.
#   - HPS >= 3.5 -> DIRECT AUTO_ACCEPT.
#   - 3.0 <= HPS < 3.5 -> AUTOMATED REVIEW.
#   - Review Score = mean of the designated compensable review factors on 0..5.
#   - Review Acceptance Threshold = midpoint(3.0, 3.5) = 3.25/5.
#       Review Score >= 3.25 -> ACCEPT AFTER REVIEW; otherwise -> AUTO_REJECT.
#   - Human reviewers verify evidence only; admission remains server-automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Uses common technical checks: schema, datatype consistency, required ranges,
#     missingness, duplicate/ID integrity, label/domain validity, and structure.
#   - Uses GX native severity-aware validation: zero critical failures required;
#     no ranking and no forced-K.
#
# Runtime/reporting:
#   - Every FL round prints configuration/run/scenario/round progress plus TRAIN-only
#     diagnostic utility and operational metrics.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v20.0
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

DIMENSION_MIN_FLOOR = 2.5
REVIEW_ACCEPT_THRESHOLD = (GOOD_CUT + HIGH_CUT) / 2.0  # 3.25 on the 0..5 rubric scale

# ----------------------------------------------------------------------
# v20.0 FINAL FULLY AUTOMATED ADMISSION POLICY
# ----------------------------------------------------------------------
# Human reviewers verify supporting evidence uploaded through the questionnaire.
# They do NOT make the admission decision. Once verified factor scores are
# available, the server applies this policy automatically.
#
# Decision order:
#   1) MANDATORY GOVERNANCE GATE:
#      every domain-required mandatory factor must be present, finite, and meet
#      its factor-specific minimum. Any failure -> AUTO_REJECT. No compensation.
#   2) DIMENSION FLOOR:
#      every averaged HPS dimension must be >= 2.5/5.
#   3) HPS:
#      HPS < 3.0 -> AUTO_REJECT.
#      HPS >= 3.5 -> DIRECT AUTO_ACCEPT.
#      3.0 <= HPS < 3.5 -> AUTOMATED REVIEW.
#   4) REVIEW:
#      Review Score = arithmetic mean of the designated compensable review
#      factors on the same 0..5 scale. Missing/non-finite review evidence
#      contributes 0 to the Review Score but is not itself a hard veto.
#      Review Score >= 3.25 -> ACCEPT AFTER REVIEW.
#      Review Score <  3.25 -> AUTO_REJECT.
#
# Thus non-negotiable eligibility and graded trustworthiness are separated:
# mandatory requirements cannot be compensated, while the Review Score is used
# only to resolve the HPS borderline interval.

MANDATORY_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "sensitivity_classification",
        ],
        "dim6": [
            "license_terms",
        ],
    },
    # Cross-domain support is retained for companion CIFAR experiments. These
    # requirements are intentionally narrower than healthcare requirements.
    "cifar10": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim6": [
            "license_terms",
        ],
    },
}

# Compensable documentary evidence used only when 3.0 <= HPS < 3.5.
# Mandatory factors are deliberately excluded so the same evidence does not
# serve simultaneously as a hard veto and as compensable review evidence.
REVIEW_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "source_reputation",
            "data_objective",
        ],
        "dim3": [
            "data_dictionary",
            "version_logs",
            "collection_protocol",
            "definition_updates",
        ],
        "dim4": [
            "data_freshness",
            "scheduled_refresh",
            "retention_clarity",
        ],
        "dim5": [
            "geo_restrictions",
            "audits",
        ],
        "dim6": [
            "ethical_reviews",
            "redistribution",
            "user_agreements",
        ],
    },
    "cifar10": {
        "dim1": [
            "source_reputation",
            "data_objective",
        ],
        "dim3": [
            "data_dictionary",
            "version_logs",
            "collection_protocol",
            "definition_updates",
        ],
        "dim4": [
            "data_freshness",
            "scheduled_refresh",
            "retention_clarity",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "geo_restrictions",
            "sensitivity_classification",
            "audits",
        ],
        "dim6": [
            "ethical_reviews",
            "redistribution",
            "user_agreements",
        ],
    },
}

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# Factor-count notes:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# two reviewer-driven factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v20.0
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    v20.0 uses pre-specified governance archetypes aligned with the revised policy:

      - DIRECT_STRONG:
          passes the Mandatory Gate and is designed for HPS >= 3.5;
      - REVIEW_RECOVERABLE:
          passes the Mandatory Gate, is designed for the HPS review band, and
          has Review Score >= 3.25;
      - REVIEW_LIMITED:
          passes the Mandatory Gate, is designed for the HPS review band, but
          has Review Score < 3.25;
      - MANDATORY_GATE_FAIL:
          otherwise strong evidence but one mandatory factor is deliberately
          one rubric level below its required minimum;
      - LOW_HPS_WEAK:
          passes the Mandatory Gate and dimension floor, but is designed for
          HPS < 3.0;
      - DIMENSION_FLOOR_WEAK:
          passes the Mandatory Gate but contains a deliberately weak
          non-mandatory dimension (<2.5).

    IMPORTANT SCIENTIFIC INTERPRETATION
    -----------------------------------
    These are controlled governance scenarios, not observed hospital-site
    provenance records and not estimates of real-world admission prevalence.
    The archetype composition is specified BEFORE model training and does not
    use predictive performance, test labels, or downstream model outcomes.

    DQ (dim2) is NEVER synthesized here; it remains measured from TRAIN data.
    The frozen evidence seed randomly assigns the pre-generated archetypes to
    client identities.
    """
    domain = str(domain).lower()
    if domain not in MANDATORY_FACTORS_BY_DOMAIN:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    mandatory_policy = MANDATORY_FACTORS_BY_DOMAIN[domain]
    review_policy = REVIEW_FACTORS_BY_DOMAIN[domain]

    mandatory_keys = [
        (dim, factor)
        for dim, factor_list in mandatory_policy.items()
        for factor in factor_list
    ]
    review_keys = [
        (dim, factor)
        for dim, factor_list in review_policy.items()
        for factor in factor_list
    ]
    mandatory_key_set = set(mandatory_keys)
    review_key_set = set(review_keys)

    def make_documentary_profile(role: str, variant: int):
        factors = {
            dim: {}
            for dim in DOCUMENTARY_DIMS
        }

        if role == "DIRECT_STRONG":
            # Strong but not uniformly perfect. Every documentary factor is at
            # least 4, and all mandatory factors therefore clear their minima.
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    minimum = float(factor_minima[dim][factor])
                    base = max(4.0, minimum)
                    bonus = 1.0 if ((j + variant + len(dim)) % 3 == 0) else 0.0
                    factors[dim][factor] = float(min(5.0, base + bonus))

        elif role == "REVIEW_RECOVERABLE":
            # Start at 3 throughout. Mandatory factors are raised to their exact
            # policy minima. The documentation-practice review factors are raised
            # to 4, giving a compensable Review Score above the 3.25 threshold
            # while keeping the overall evidence near the HPS review band.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for dim, factor in mandatory_keys:
                factors[dim][factor] = float(factor_minima[dim][factor])

            # Four documentation-practice factors at 4 are sufficient to put the
            # 14-factor healthcare Review Score just above 3.25:
            # (10*3 + 4*4)/14 = 3.2857.
            for factor in FACTOR_NAMES["dim3"]:
                if ("dim3", factor) in review_key_set:
                    factors["dim3"][factor] = 4.0

            # Small deterministic variation without crossing mandatory minima.
            if variant % 2 == 1:
                factors["dim4"]["scheduled_refresh"] = 4.0
                factors["dim6"]["user_agreements"] = 2.0

        elif role == "REVIEW_LIMITED":
            # Pass all mandatory requirements and keep dimensions >=2.5, but hold
            # the compensable review factors at 3.0 so Review Score = 3.0 < 3.25.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for dim, factor in mandatory_keys:
                factors[dim][factor] = float(factor_minima[dim][factor])

        elif role == "MANDATORY_GATE_FAIL":
            # Otherwise strong evidence, with exactly one policy-mandatory factor
            # deliberately one rubric level below its minimum. HPS/dimensions
            # remain strong so rejection is attributable to the Mandatory Gate.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    minimum = float(factor_minima[dim][factor])
                    factors[dim][factor] = float(max(4.0, minimum))

            weak_key = mandatory_keys[variant % len(mandatory_keys)]
            weak_min = float(factor_minima[weak_key[0]][weak_key[1]])
            factors[weak_key[0]][weak_key[1]] = float(max(0.0, weak_min - 1.0))

        elif role == "LOW_HPS_WEAK":
            # Construct low but dimension-floor-safe evidence. Mandatory factors
            # are then raised to their exact minima. This isolates the HPS<3.0
            # path as far as TRAIN-measured DQ permits.
            template = {
                "dim1": [2.0, 4.0, 2.0, 4.0],  # average 3.0
                "dim3": [2.0, 3.0, 2.0, 3.0],  # average 2.5
                "dim4": [2.0, 3.0, 3.0],       # average 2.667
                "dim5": [3.0, 3.0, 2.0, 3.0, 2.0],  # average 2.6
                "dim6": [3.0, 2.0, 2.0, 3.0],       # average 2.5
            }
            for dim in DOCUMENTARY_DIMS:
                for factor, value in zip(FACTOR_NAMES[dim], template[dim]):
                    factors[dim][factor] = float(value)

            for dim, factor in mandatory_keys:
                factors[dim][factor] = max(
                    float(factors[dim][factor]),
                    float(factor_minima[dim][factor]),
                )

        elif role == "DIMENSION_FLOOR_WEAK":
            # Mandatory requirements pass; Timeliness is intentionally <2.5.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for dim, factor in mandatory_keys:
                factors[dim][factor] = float(factor_minima[dim][factor])

            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # For K=10, preserve the old bundle identity structure E01..E10 so the
    # frozen evidence seed maps the same bundle numbers to the same client IDs.
    # The revised branch-coverage set contains:
    #   E01-E04: 4 direct-strong
    #   E05-E06: 2 review-recoverable
    #   E07:     1 mandatory-gate failure
    #   E08:     1 review-limited
    #   E09:     1 low-HPS weak
    #   E10:     1 dimension-floor weak
    #
    # This design targets six admissible bundles and four rejected bundles while
    # explicitly exercising the new Mandatory Gate.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("MANDATORY_GATE_FAIL", 0),
            ("REVIEW_LIMITED", 0),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "MANDATORY_GATE_FAIL",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        factors = make_documentary_profile(role, int(variant))
        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    assignment = rng.permutation(n)
    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]

        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                min_rank = float(factor_minima[dim][factor])
                is_mandatory = (dim, factor) in mandatory_key_set
                is_review = (dim, factor) in review_key_set

                rows.append({
                    "client": str(cid),
                    "bundle_id": bundle["bundle_id"],
                    "evidence_profile": bundle["profile"],
                    "scenario_role": bundle["scenario_role"],
                    "profile_variant": int(bundle["profile_variant"]),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "evidence_source_type": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5": float(score),
                    "rubric_score_0_5": float(score),
                    "server_mapped_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "human_role": "VERIFY_EVIDENCE_ONLY",
                    "score_assignment": "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by": "SERVER_POLICY",
                    "adequacy_min_rank": min_rank,
                    "adequacy_min_normalized": min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy": bool(float(score) >= min_rank),
                    "is_mandatory_factor": bool(is_mandatory),
                    "mandatory_min_rank": min_rank if is_mandatory else np.nan,
                    "meets_mandatory_requirement": (
                        bool(float(score) >= min_rank) if is_mandatory else np.nan
                    ),
                    "is_review_factor": bool(is_review),
                    "review_acceptance_threshold_0_5": (
                        float(REVIEW_ACCEPT_THRESHOLD) if is_review else np.nan
                    ),
                    "evidence_artifact_id": f"{cid}-{bundle['bundle_id']}-{dim}-{factor}",
                    "validation_status": "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed": int(evidence_seed),
                    "domain": domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client-specific HPS dimension scores (0..5):
    mean of the observed factor rubric scores within each dimension.
    """
    out = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        out[dim] = float(np.mean(vals)) if vals else 0.0
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def all_zero_score_factors(
    factors: Dict[str, Dict[str, float]]
) -> List[str]:
    """Audit-only list of finite zero-score factors."""
    zeros = []
    for dim in FACTOR_NAMES:
        for factor, value in factors.get(dim, {}).items():
            if np.isfinite(float(value)) and float(value) <= 0.0:
                zeros.append(f"{dim}.{factor}")
    return zeros


def mandatory_gate_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Evaluate the non-compensable Mandatory Governance Gate.

    A mandatory requirement passes only when the verified factor score is
    present, finite, and >= its factor-specific policy minimum.
    """
    domain = str(domain).lower()
    policy = MANDATORY_FACTORS_BY_DOMAIN[domain]
    minima = DOMAIN_FACTOR_MINIMA[domain]

    scores = {}
    minimums = {}
    missing = []
    below_minimum = []

    for dim, factor_list in policy.items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            threshold = float(minima[dim][factor])
            minimums[key] = threshold

            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            value = float(factors[dim][factor])
            if not np.isfinite(value):
                missing.append(key)
                continue

            scores[key] = value
            if value < threshold:
                below_minimum.append(
                    f"{key}:{value:.1f}<{threshold:.1f}"
                )

    expected_n = sum(len(v) for v in policy.values())
    passed = (
        len(missing) == 0
        and len(scores) == expected_n
        and len(below_minimum) == 0
    )

    return {
        "passed": bool(passed),
        "scores": scores,
        "minimums": minimums,
        "missing": missing,
        "below_minimum": below_minimum,
        "mandatory_factor_count": int(expected_n),
    }


def review_score_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Compute the compensable Review Score on the original 0..5 rubric scale.

    Only designated non-mandatory documentary factors are included.
    Missing/non-finite review evidence contributes 0.0 rather than acting as a
    hard veto. The score is used ONLY when 3.0 <= HPS < 3.5.
    """
    domain = str(domain).lower()
    policy = REVIEW_FACTORS_BY_DOMAIN[domain]

    scores = {}
    missing = []
    values = []

    for dim, factor_list in policy.items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            if factor not in factors.get(dim, {}):
                missing.append(key)
                scores[key] = 0.0
                values.append(0.0)
                continue

            value = float(factors[dim][factor])
            if not np.isfinite(value):
                missing.append(key)
                scores[key] = 0.0
                values.append(0.0)
                continue

            value = float(np.clip(value, 0.0, MAX_FACTOR_SCORE))
            scores[key] = value
            values.append(value)

    if not values:
        raise ValueError(f"No Review Score factors configured for domain={domain!r}")

    score = float(np.mean(values))
    return {
        "review_score_0_5": score,
        "review_acceptance_threshold_0_5": float(REVIEW_ACCEPT_THRESHOLD),
        "review_score_margin": float(score - REVIEW_ACCEPT_THRESHOLD),
        "review_factor_count": int(len(values)),
        "scores": scores,
        "missing": missing,
    }


def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return dimensions that are missing/non-finite or below the policy floor."""
    return [
        dim
        for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def factor_adequacy_attainment(
    factors: Dict[str, Dict[str, float]],
    factor_minima: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    total = 0
    passed = 0
    per_dim = {}

    for dim in FACTOR_NAMES:
        dim_total = 0
        dim_passed = 0

        for factor in FACTOR_NAMES[dim]:
            total += 1
            dim_total += 1

            score = float(factors[dim][factor])
            threshold = float(factor_minima[dim][factor])

            if score >= threshold:
                passed += 1
                dim_passed += 1

        per_dim[dim] = {
            "passed": dim_passed,
            "total": dim_total,
            "fraction": float(dim_passed / max(1, dim_total)),
        }

    return {
        "passed": passed,
        "total": total,
        "fraction": float(passed / max(1, total)),
        "per_dim": per_dim,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    v20.1 final fully automated admission policy.

    Flow:
      Mandatory Governance Gate:
          any required factor missing/non-finite/below its policy minimum
          -> AUTO_REJECT (no compensation)

      EVERY dimension >= 2.5?
          NO -> AUTO_REJECT

      HPS < 3.0?
          YES -> AUTO_REJECT

      HPS >= 3.5?
          YES -> DIRECT AUTO_ACCEPT

      3.0 <= HPS < 3.5:
          Review Score >= 3.25 -> ACCEPT AFTER AUTOMATED REVIEW
          otherwise            -> AUTO_REJECT
    """
    domain = str(domain).lower()

    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)

    attainment = factor_adequacy_attainment(
        factors,
        factor_minima,
    )
    zeros = all_zero_score_factors(factors)
    mandatory = mandatory_gate_audit(factors, domain)
    review = review_score_audit(factors, domain)

    dim_floor_failed = dimension_floor_failures(
        dims,
        floor=DIMENSION_MIN_FLOOR,
    )

    mandatory_score_string = ";".join(
        f"{key}={value:.1f}(min={mandatory['minimums'][key]:.1f})"
        for key, value in mandatory["scores"].items()
    )
    review_score_string = ";".join(
        f"{key}={value:.1f}"
        for key, value in review["scores"].items()
    )

    base = {
        "hps": float(hps),
        "dimensions": dims,
        "dimension_min_floor": float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures": ";".join(dim_floor_failed),

        "mandatory_gate_pass": bool(mandatory["passed"]),
        "mandatory_factor_count": int(mandatory["mandatory_factor_count"]),
        "mandatory_scores": mandatory_score_string,
        "mandatory_missing": ";".join(mandatory["missing"]),
        "mandatory_below_minimum": ";".join(mandatory["below_minimum"]),

        "review_score_0_5": float(review["review_score_0_5"]),
        "review_acceptance_threshold_0_5": float(
            review["review_acceptance_threshold_0_5"]
        ),
        "review_score_margin": float(review["review_score_margin"]),
        "review_factor_count": int(review["review_factor_count"]),
        "review_scores": review_score_string,
        "review_missing": ";".join(review["missing"]),

        "factor_adequacy_passed": int(attainment["passed"]),
        "factor_adequacy_total": int(attainment["total"]),
        "factor_adequacy_fraction": float(attainment["fraction"]),
        "all_zero_factors": ";".join(zeros),
    }

    # Gate 1: non-negotiable governance requirements.
    if not mandatory["passed"]:
        failures = mandatory["missing"] + mandatory["below_minimum"]
        return {
            **base,
            "decision_path": "MANDATORY_GOVERNANCE_GATE",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_MANDATORY_GATE",
            "reason": (
                "Mandatory governance requirement failed: "
                + ";".join(failures)
            ),
        }

    # Gate 2: every complete trustworthiness dimension must clear 2.5/5.
    if dim_floor_failed:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Gate 3: overall HPS lower bound.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path": "LOW_HPS",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_LOW_HPS",
            "reason": f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # Clear high-HPS region after mandatory and dimension gates have passed.
    if hps >= float(high_cut):
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "initial_action": "AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "status": "DIRECT_AUTO_ACCEPTED",
            "reason": (
                f"Mandatory Gate passed; all dimensions >= "
                f"{DIMENSION_MIN_FLOOR:.1f}; HPS {hps:.3f} >= {high_cut:.3f}"
            ),
        }

    # Borderline HPS region: resolve with the simple Review Score.
    review_pass = bool(
        float(review["review_score_0_5"]) >= float(REVIEW_ACCEPT_THRESHOLD)
    )

    if review_pass:
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "initial_action": "AUTOMATED_REVIEW",
            "final_action": "ACCEPT",
            "status": "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"HPS {hps:.3f} in [{good_cut:.3f}, {high_cut:.3f}); "
                f"Review Score {review['review_score_0_5']:.3f} >= "
                f"Review Acceptance Threshold {REVIEW_ACCEPT_THRESHOLD:.3f}"
            ),
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "initial_action": "AUTOMATED_REVIEW",
        "final_action": "REJECT",
        "status": "AUTO_REJECTED_REVIEW_SCORE",
        "reason": (
            f"HPS {hps:.3f} in [{good_cut:.3f}, {high_cut:.3f}); "
            f"Review Score {review['review_score_0_5']:.3f} < "
            f"Review Acceptance Threshold {REVIEW_ACCEPT_THRESHOLD:.3f}"
        ),
    }

def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }

        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(
            factors,
            domain=domain,
        )

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]),

            "mandatory_gate_pass": bool(d["mandatory_gate_pass"]),
            "mandatory_factor_count": int(d["mandatory_factor_count"]),
            "mandatory_scores": d["mandatory_scores"],
            "mandatory_missing": d["mandatory_missing"],
            "mandatory_below_minimum": d["mandatory_below_minimum"],

            "review_score_0_5": float(d["review_score_0_5"]),
            "review_acceptance_threshold_0_5": float(
                d["review_acceptance_threshold_0_5"]
            ),
            "review_score_margin": float(d["review_score_margin"]),
            "review_factor_count": int(d["review_factor_count"]),
            "review_scores": d["review_scores"],
            "review_missing": d["review_missing"],

            "dimension_min_floor": float(d["dimension_min_floor"]),
            "dimension_floor_failures": d["dimension_floor_failures"],
            "factor_adequacy_passed": int(d["factor_adequacy_passed"]),
            "factor_adequacy_total": int(d["factor_adequacy_total"]),
            "factor_adequacy_fraction": float(d["factor_adequacy_fraction"]),
            "decision_path": d["decision_path"],
            "initial_action": d["initial_action"],
            "final_action": d["final_action"],
            "status": d["status"],
            "reason": d["reason"],
            "all_zero_factors": d["all_zero_factors"],
        }

        for dim, value in d["dimensions"].items():
            row[f"{dim}_score_0_5"] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(
    governance_df: pd.DataFrame
) -> List[str]:
    """
    Select one best TADP-eligible client for SDA from already accepted clients.
    """
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "TADP-SDA cannot select a client: no TADP-eligible client."
        )

    eligible["direct_accept_priority"] = (
        eligible["status"]
        .eq("DIRECT_AUTO_ACCEPTED")
        .astype(int)
    )

    eligible = eligible.sort_values(
        [
            "hps",
            "direct_accept_priority",
            "review_score_0_5",
            "client",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return [str(eligible.iloc[0]["client"])]

def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit.

    v20.0 explicitly labels policy-mandatory factors and compensable Review
    Score factors. The Mandatory Gate uses factor-specific policy minima;
    Review Score factors are used only in the HPS borderline band.
    """
    domain = str(domain).lower()
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    mandatory_policy = MANDATORY_FACTORS_BY_DOMAIN[domain]
    review_policy = REVIEW_FACTORS_BY_DOMAIN[domain]

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index("client", drop=False)

    def policy_fields(dim, factor):
        is_mandatory = bool(
            factor in mandatory_policy.get(dim, [])
        )
        is_review = bool(
            factor in review_policy.get(dim, [])
        )
        return {
            "is_mandatory_factor": is_mandatory,
            "mandatory_min_rank": (
                float(factor_minima[dim][factor])
                if is_mandatory else np.nan
            ),
            "is_review_factor": is_review,
            "review_acceptance_threshold_0_5": (
                float(REVIEW_ACCEPT_THRESHOLD)
                if is_review else np.nan
            ),
        }

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(documentary_evidence[cid][dim][factor])
                min_rank = float(factor_minima[dim][factor])

                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value": np.nan,
                    "rubric_score_0_5": score,
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "adequacy_min_rank": min_rank,
                    "adequacy_min_normalized": min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy": bool(score >= min_rank),
                    **policy_fields(dim, factor),
                    "evidence_seed": int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(dq_scores[cid][factor])
            min_rank = float(factor_minima["dim2"][factor])
            raw_col = DQ_RAW_METRIC_BY_FACTOR[factor]

            raw_value = (
                float(dq_index.loc[cid, raw_col])
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name": DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source": "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value": raw_value,
                "rubric_score_0_5": score,
                "rubric_descriptor": rubric_descriptor("dim2", factor, score),
                "adequacy_min_rank": min_rank,
                "adequacy_min_normalized": min_rank / MAX_FACTOR_SCORE,
                "meets_factor_adequacy": bool(score >= min_rank),
                **policy_fields("dim2", factor),
                "evidence_seed": int(evidence_seed),
            })

    out = pd.DataFrame(rows)
    expected_rows = len(client_ids) * 28

    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain "
            f"{expected_rows} rows; found {len(out)}."
        )

    return out

# ======================================================================================
# GREAT EXPECTATIONS# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO TADP governance signal in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096
ROUND_PROGRESS_POWER_W = 12.0


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Cumulative model traffic through this round. Same accounting as the
        # final experiment metric: download + upload + 12% protocol overhead.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(
            (cumulative_comm_raw_b * 1.12) / (1024 ** 2)
        )

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b * 1.12)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }



def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Select top-K clients by the machine-measured TADP Data Quality dimension only."""
    rows = []
    for cid, scores in dq_scores.items():
        vals = [float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client": str(cid),
            "dq_only_score": float(np.mean(vals)),
            "dq_factor_count": int(len(vals)),
        })
    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score", "client"], ascending=[False, True]
    ).reset_index(drop=True)
    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    return selected, audit


def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Power-of-Choice baseline: each round samples d candidates and selects the K
    clients with largest current local TRAIN loss. K, rounds, initialisation, and
    total optimizer steps per round are matched to TADP-VR. No TEST information is used.
    """
    monitor = RAMMonitor().start()
    start = time.perf_counter()
    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    rng = np.random.default_rng(int(run_seed) + 880000)
    round_rows = []
    total_steps = 0
    total_selected = 0
    communication_b = 0
    monitor_eval_s = 0.0

    all_candidates = list(candidate_clients)
    if int(select_k) < 1 or int(select_k) > len(all_candidates):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1, NUM_ROUNDS_FL + 1):
        d = min(len(all_candidates), max(int(select_k), int(candidate_multiplier) * int(select_k)))
        if d == len(all_candidates):
            candidate_pool = list(all_candidates)
        else:
            # Canonical pow-d samples candidate clients without replacement
            # according to p_k, the client's fraction of total TRAIN data.
            sizes = np.asarray(
                [len(client_arrays[c][1]) for c in all_candidates], dtype=float
            )
            probs = sizes / sizes.sum()
            candidate_pool = rng.choice(
                all_candidates, size=d, replace=False, p=probs
            ).tolist()

        losses = []
        for cid in candidate_pool:
            Xc, yc = client_arrays[cid]
            losses.append((str(cid), local_training_loss(global_model, Xc, yc)))
        losses.sort(key=lambda x: (-x[1], x[0]))
        selected = [cid for cid, _ in losses[:int(select_k)]]
        step_map = allocate_exact_step_budget(
            selected, client_arrays, int(target_steps_per_round), batch_size, local_epochs
        )

        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights, local_sizes = [], []
        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]), batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes, equal_weight=False)
        global_model = build_model_fn()
        global_model.set_weights(agg)
        round_steps = int(sum(step_map.values()))
        total_steps += round_steps
        total_selected += len(selected)
        param_b = model_parameter_bytes(global_model)
        # Candidate clients receive the current model to evaluate local loss;
        # selected clients return one model update. Scalar loss uploads are negligible.
        communication_b += (len(candidate_pool) + len(selected)) * param_b
        cumulative_comm_mb = float(
            (communication_b * 1.12) / (1024 ** 2)
        )
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_rows.append({
            "round": r,
            "candidate_count": len(candidate_pool),
            "candidate_ids": ";".join(candidate_pool),
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": round_steps,
            "cumulative_optimizer_steps": int(total_steps),
            "local_losses": json.dumps({cid: loss for cid, loss in losses}, sort_keys=True),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        })

        loss_preview = ", ".join(
            f"{cid}:{loss:.3f}" for cid, loss in losses[:min(5, len(losses))]
        )
        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=NUM_ROUNDS_FL,
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
            extra=f"PoC candidates={len(candidate_pool)} | highest TRAIN losses: {loss_preview}",
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()
    communication_b = int(communication_b * 1.12)
    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(total_selected / max(1, NUM_ROUNDS_FL)),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_mb": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.430)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_diabetes_no_leakage(
    csv_path: str,
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, split_seed
    )

    clients = dirichlet_partition_dataframe(
        train_df, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(clients.keys())

    # Hard row-level leakage audit.
    client_train_ids = {
        cid: df["_row_id"].astype(str).to_numpy()
        for cid, df in clients.items()
    }

    pre = fit_federated_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)

    dq_scores = {}
    dq_raw_rows = []
    client_arrays = {}

    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores

        row = {"client": cid, **scores, **raw_metrics}
        dq_raw_rows.append(row)

        X = pre.transform(df)
        y = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (X, y)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train_all = np.concatenate(
        [client_arrays[c][1] for c in client_ids]
    )
    cw = class_weight_dict(y_train_all)

    meta = {
        "raw_rows": len(raw),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "patient_overlap": int(patient_overlap),
        "input_dim": int(X_test.shape[1]),
        "n_clients": int(n_clients),
        "numeric_features": len(pre.numeric_cols),
        "categorical_features": len(pre.categorical_cols),
    }

    return {
        "raw": raw,
        "train_df": train_df,
        "test_df": test_df,
        "clients_raw": clients,
        "client_arrays": client_arrays,
        "client_ids": client_ids,
        "X_test": X_test,
        "y_test": y_test,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_raw_rows),
        "class_weights": cw,
        "meta": meta,
        "client_train_ids": client_train_ids,
        "global_train_ids": train_df["_row_id"].astype(str).to_numpy(),
        "global_test_ids": test_df["_row_id"].astype(str).to_numpy(),
        "preprocessor": pre,
        "dq_reference": reference,
    }


def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# These eight configurations are genuinely distinct and are repeated over all
# five training seeds for mean ± SD / CI reporting.
#
# NOT repeated here:
#   - Great Expectations Centralized
#   - TADP-AA Centralized
#   - Great Expectations Federated
#   - TADP-AA Federated
# Those all-client equivalences are verified independently by the companion
# one-seed equivalence-audit script.
#
# TADP-SDA is retained as a boundary/stress condition but is run once only,
# outside the five-seed headline statistical comparison.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8

BOUNDARY_SCENARIOS = [
    "TADP-SDA Centralized",
    "TADP-SDA Federated",
]
BOUNDARY_SEED = 42



def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarize the frozen controlled documentary-evidence assignment.

    v20.0 reports the pre-specified governance archetype for each evidence
    bundle. These archetypes are branch-coverage scenarios, not observed
    real-world prevalence classes.
    """
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "meets_factor_adequacy",
        "evidence_seed",
    }

    missing = sorted(
        required - set(evidence_df.columns)
    )

    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["meets_factor_adequacy"] = (
        work["meets_factor_adequacy"]
        .astype(bool)
    )

    summary = (
        work
        .groupby(
            ["client", "bundle_id"],
            as_index=False,
        )
        .agg(
            evidence_profile=(
                "evidence_profile",
                "first",
            ),
            scenario_role=(
                "scenario_role",
                "first",
            ),
            profile_variant=(
                "profile_variant",
                "first",
            ),
            documentary_factor_count=(
                "factor",
                "count",
            ),
            documentary_adequate_factor_count=(
                "meets_factor_adequacy",
                "sum",
            ),
            documentary_mean_score=(
                "rubric_score_0_5",
                "mean",
            ),
            documentary_min_score=(
                "rubric_score_0_5",
                "min",
            ),
            documentary_max_score=(
                "rubric_score_0_5",
                "max",
            ),
            evidence_seed=(
                "evidence_seed",
                "first",
            ),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    summary["documentary_adequacy_fraction"] = (
        summary["documentary_adequate_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d])
        for d in DOCUMENTARY_DIMS
    )

    count_ok = summary[
        "documentary_factor_count"
    ].eq(
        expected_documentary_factors
    )

    if not count_ok.all():
        bad = summary.loc[
            ~count_ok,
            [
                "client",
                "bundle_id",
                "documentary_factor_count",
            ],
        ]

        raise RuntimeError(
            "Unexpected controlled-evidence factor count. "
            f"Expected {expected_documentary_factors} documentary factors "
            "per client. Offending rows:\n"
            + bad.to_string(index=False)
        )

    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    domain = str(domain).lower()

    print_banner("DOMAIN ADMISSION POLICY — MANDATORY GATE + HPS + REVIEW SCORE")

    mandatory_rows = []
    for dim, factor_list in MANDATORY_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            minimum = float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
            mandatory_rows.append({
                "dimension": dim,
                "dimension_name": DIMENSION_NAMES[dim],
                "factor": factor,
                "mandatory_minimum_0_5": minimum,
                "rule": f"score >= {minimum:.1f}; missing/non-finite also fails",
            })

    print("\nMANDATORY GOVERNANCE REQUIREMENTS")
    print(pd.DataFrame(mandatory_rows).to_string(index=False))

    review_rows = []
    for dim, factor_list in REVIEW_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            review_rows.append({
                "dimension": dim,
                "dimension_name": DIMENSION_NAMES[dim],
                "factor": factor,
            })

    print("\nCOMPENSABLE REVIEW FACTORS")
    print(pd.DataFrame(review_rows).to_string(index=False))

    print(
        f"\nMinimum dimension floor: EVERY averaged dimension "
        f"must be >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )
    print(
        f"HPS policy: <{GOOD_CUT:.1f}=Reject; "
        f"[{GOOD_CUT:.1f},{HIGH_CUT:.1f})=Automated Review; "
        f">={HIGH_CUT:.1f}=Direct Accept."
    )
    print(
        f"Review Acceptance Threshold: {REVIEW_ACCEPT_THRESHOLD:.2f}/5 "
        f"(midpoint of {GOOD_CUT:.1f} and {HIGH_CUT:.1f})."
    )
    print(
        "Human reviewer role: verify uploaded questionnaire evidence only. "
        "The server makes the admission decision automatically."
    )

    print_banner("TRAIN-ONLY DATA-QUALITY FACTORS — 8 FACTORS")
    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]
    print(dq[dq_cols].to_string(index=False))

    print_banner("FROZEN TADP GOVERNANCE — MANDATORY GATE + HPS + REVIEW SCORE")
    display_cols = [
        "client",
        "mandatory_gate_pass",
        "mandatory_below_minimum",
        "hps",
        "review_score_0_5",
        "review_acceptance_threshold_0_5",
        "review_score_margin",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]
    print(gov[display_cols].to_string(index=False))

    print("\nTADP v20.0 final decision order:")
    print("  1) Mandatory Governance Gate:")
    print("       any required factor missing/non-finite/below its policy minimum -> AUTO-REJECT")
    print(
        f"  2) ANY averaged dimension < {DIMENSION_MIN_FLOOR:.1f}/5 -> AUTO-REJECT"
    )
    print(f"  3) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT")
    print(f"  4) HPS >= {HIGH_CUT:.1f} -> DIRECT AUTO-ACCEPT")
    print(
        f"  5) {GOOD_CUT:.1f} <= HPS < {HIGH_CUT:.1f} -> AUTOMATED REVIEW"
    )
    print(
        f"       Review Score >= {REVIEW_ACCEPT_THRESHOLD:.2f}/5 "
        "-> ACCEPT AFTER REVIEW"
    )
    print("       otherwise -> AUTO-REJECT")

    print(
        f"\nFrozen TADP-VR cohort: "
        f"{len(vr_clients)}/10 -> {vr_clients}"
    )
    print(
        f"Frozen TADP-SDA cohort: "
        f"{len(sda_clients)}/10 -> {sda_clients}"
    )

    print_banner("GX CORE NATIVE TRAIN-ONLY VALIDATOR BASELINE")
    gx_base_cols = [
        "client", "gx_native_suite_success", "ge_expectations_passed",
        "ge_expectations_total", "ge_pass_rate", "ge_native_validation_class",
        "ge_final_action",
    ]
    gx_category_cols = [
        c for c in ge.columns
        if c.startswith("ge_") and (c.endswith("_passed") or c.endswith("_total"))
        and c not in {"ge_expectations_passed", "ge_expectations_total"}
    ]
    print(ge[gx_base_cols + sorted(gx_category_cols)].to_string(index=False))
    print(
        f"\nGX operationally valid clients (zero critical failures): "
        f"{len(ge_clients)}/{len(ge)} -> {ge_clients}"
    )
    print(
        "GX is used only as a native rule-based data validator. PASS/FAIL is the "
        "suite-level GX result; no ranking, no forced-K selection, and no TADP signal is used."
    )

def build_hash_chained_governance_ledger(
    gov,
    ge,
    output_path,
):
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in gov.sort_values("client").iterrows():
        seq += 1
        payload = {
            "sequence": seq,
            "governance_system": "TADP",
            "client": str(r["client"]),
            "hps": float(r["hps"]),
            "mandatory_gate_pass": bool(r["mandatory_gate_pass"]),
            "mandatory_below_minimum": str(r["mandatory_below_minimum"]),
            "review_score_0_5": float(r["review_score_0_5"]),
            "review_acceptance_threshold_0_5": float(
                r["review_acceptance_threshold_0_5"]
            ),
            "review_score_margin": float(r["review_score_margin"]),
            "dimension_floor_failures": str(r["dimension_floor_failures"]),
            "decision_path": str(r["decision_path"]),
            "initial_action": str(r["initial_action"]),
            "final_action": str(r["final_action"]),
            "status": str(r["status"]),
            "reason": str(r["reason"]),
            "previous_hash": prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )
        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()
        payload["entry_hash"] = entry_hash
        rows.append(payload)
        prev_hash = entry_hash

    for _, r in ge.sort_values("client").iterrows():
        seq += 1
        payload = {
            "sequence": seq,
            "governance_system": "Great Expectations GX Core native validator",
            "client": str(r["client"]),
            "hps": None,
            "mandatory_gate_pass": None,
            "mandatory_below_minimum": None,
            "review_score_0_5": None,
            "review_acceptance_threshold_0_5": None,
            "review_score_margin": None,
            "dimension_floor_failures": None,
            "decision_path": "RULE_BASED_VALIDATION",
            "initial_action": "RULE_BASED_VALIDATION",
            "final_action": str(r["ge_final_action"]),
            "status": "GX_CONTROLLED_TRAIN_ONLY",
            "reason": (
                f"GX native suite {'PASSED' if bool(r['gx_native_suite_success']) else 'FAILED'}; "
                f"{int(r['ge_expectations_passed'])}/{int(r['ge_expectations_total'])} "
                "configured technical Expectations passed"
            ),
            "previous_hash": prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )
        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()
        payload["entry_hash"] = entry_hash
        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)
    out.to_csv(output_path, index=False)
    return out

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized","centralized","baseline","all clients","5-seed headline"],
        ["Great Expectations Centralized","centralized","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["TADP-AA Centralized","centralized","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Centralized","centralized","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Centralized","centralized","TADP-SDA","frozen best eligible client","one-seed boundary"],
        ["Vanilla FedAvg","federated","FedAvg","all clients","5-seed headline"],
        ["FedProx","federated","FedProx","all clients","5-seed headline"],
        ["Random-K","federated","matched random control","same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Great Expectations Federated","federated","GX native validator","GX clients with zero critical failures","equivalence audit only"],
        ["DQ-only Federated","federated","DQ-only","top-K by machine-measured DQ; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["Power-of-Choice","federated","Power-of-Choice","dynamic loss-based selection; same K/rounds/steps as TADP-VR","5-seed headline"],
        ["TADP-AA Federated","federated","TADP-AA","all clients","equivalence audit only"],
        ["TADP-VR Federated","federated","TADP-VR","frozen TADP-VR cohort","5-seed headline"],
        ["TADP-SDA Federated","federated","TADP-SDA","frozen best eligible client","one-seed boundary"],
    ], columns=["scenario","paradigm","method","participation","execution_role"])



def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows = []
    for j in range(int(n_realizations)):
        seed = int(base_seed + j)
        evidence, _ = generate_controlled_documentary_evidence(
            client_ids,
            seed,
            DOMAIN_FACTOR_MINIMA[domain],
            domain=domain,
        )
        gov = build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted = accepted_tadp_vr(gov)
        rows.append({
            "realization": j + 1,
            "evidence_seed": seed,
            "accepted_count": len(accepted),
            "accepted_clients": ";".join(accepted),
            "mean_hps": float(gov["hps"].mean()),
            "mean_review_score_0_5": float(gov["review_score_0_5"].mean()),
            "mandatory_gate_rejects": int(
                gov["status"].eq("AUTO_REJECTED_MANDATORY_GATE").sum()
            ),
            "dimension_floor_rejects": int(
                gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()
            ),
            "low_hps_rejects": int(
                gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()
            ),
            "review_accepts": int(
                gov["status"].eq("ACCEPTED_AFTER_AUTOMATED_REVIEW").sum()
            ),
            "review_rejects": int(
                gov["status"].eq("AUTO_REJECTED_REVIEW_SCORE").sum()
            ),
            "direct_auto_accepts": int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)

# ======================================================================================
# TADP EXPERIMENT G — ADVERSARIAL DIAGNOSTIC PROBE
# ======================================================================================
# PURPOSE
# -------
# This is a deliberately scoped DIAGNOSTIC experiment for the manuscript response to
# the reviewer concern about malicious contributors. It does NOT claim that TADP is a
# poisoning/backdoor defense and it does NOT evaluate arbitrary Byzantine model-update
# replacement, server compromise, collusion, or ledger tampering. Those runtime-security
# threats remain within the TADP-Sec extension.
#
# Threat model tested here:
#   * a malicious contributor controls its own local TRAIN data and therefore the local
#     update produced from those data;
#   * it cannot alter other clients, the server, the frozen global holdout, or the ledger;
#   * mandatory documentary evidence is assumed to have been externally verified and
#     therefore cannot simply be forged into a passing mandatory factor;
#   * for the provenance-inflation stress test only, all NON-MANDATORY documentary
#     review factors are set to their maximum score as a conservative worst-case test,
#     while mandatory evidence remains genuine and Data Quality remains machine-measured.
#
# Attacks:
#   1) LABEL_FLIP: cyclic valid-label flipping on a fixed fraction of each malicious
#      client's TRAIN records. This preserves the valid label domain by construction.
#   2) BACKDOOR: a rare but schema-valid categorical trigger (derived from TRAIN only)
#      is inserted into a fixed fraction of malicious TRAIN records and their labels are
#      changed to a fixed target class. Attack success rate (ASR) is measured only on a
#      triggered copy of the held-out TEST set after training; it never affects training,
#      governance, client selection, or tuning.
#   3) PROVENANCE_INFLATION: a deliberately poor local dataset is paired with maximally
#      favorable NON-MANDATORY documentary scores. Mandatory evidence is unchanged.
#      This tests whether machine-measured DQ + the 2.5 dimension floor prevent strong
#      documentary scores from compensating for severely degraded data quality.
#
# Contamination prevalence is defined over the K=10 client population. Governance-only
# diagnostics are evaluated at 10%, 20%, and 30% (1, 2, and 3 malicious clients). To keep
# the reviewer-response experiment compact, model training is performed only at the 20%
# contamination level. Malicious clients are selected only from the CLEAN TADP-VR eligible
# cohort so the probe asks whether an otherwise eligible contributor can evade the gate.
#
# Training comparisons (20% contamination only):
#   * ALL_CLIENT: attacked FedAvg with all 10 clients (no TADP participation gate).
#   * TADP_GATE: attacked FedAvg using the TADP-VR cohort recomputed after the attack.
# Clean Accuracy/F1/AUC references are reused from the already completed Experiment A v20.1
# for the same seeds, split, model, rounds, and participation regimes. Therefore no clean
# models are retrained here. Backdoor ASR is reported as an absolute diagnostic because the
# Experiment A clean models were not retained for triggered-test evaluation.
# ======================================================================================

import copy

EXPERIMENT_VERSION = (
    "TADP-G-v20.2-ADVERSARIAL-DIAGNOSTIC-K10-3SEED-20PCT-18RUN-"
    "MANDATORYGATE-REVIEWSCORE325-EMBEDDED-CLEAN-REF"
)

TRAINING_RUN_SEEDS = [42, 142, 242]
NUM_CLIENTS = 10
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0
GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 7101
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 1042
ATTACK_CLIENT_SELECTION_SEED = 9042

CONTAMINATION_LEVELS = [0.10, 0.20, 0.30]  # governance-only diagnostic
TRAINING_CONTAMINATION_LEVEL = 0.20  # model training only
LABEL_FLIP_RECORD_FRACTION = 0.20
BACKDOOR_RECORD_FRACTION = 0.10
PROVENANCE_BAD_RECORD_FRACTION = 0.45
PROVENANCE_DUPLICATE_RECORD_FRACTION = 0.30
BACKDOOR_TARGET_CLASS = 2  # Diabetes 130-US: <30-day readmission class
BACKDOOR_TRIGGER_FEATURES = 3
DOMAIN = "healthcare"

USE_GOOGLE_DRIVE_CHECKPOINTS = True

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_G_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "adversarial_performance_checkpoint.csv"
GOV_AUDIT_PATH = EXPERIMENT_ROOT / "adversarial_governance_audit.csv"
ATTACK_AUDIT_PATH = EXPERIMENT_ROOT / "attack_data_audit.csv"

ATTACKS = ["LABEL_FLIP", "BACKDOOR", "PROVENANCE_INFLATION"]
GATE_MODES = ["ALL_CLIENT", "TADP_GATE"]

# Exact held-out TEST references from completed Experiment A v20.1.
# ALL_CLIENT = Vanilla FedAvg; TADP_GATE = TADP-VR Federated.
# These values are reused only to calculate degradation; they are not used for
# attack construction, governance, client selection, training, or tuning.
EXPERIMENT_A_CLEAN_REFERENCE = {
    42: {
        "ALL_CLIENT": {"accuracy": 0.506111, "f1_macro": 0.354530, "roc_auc_ovr_macro": 0.630415},
        "TADP_GATE": {"accuracy": 0.498593, "f1_macro": 0.318031, "roc_auc_ovr_macro": 0.629645},
    },
    142: {
        "ALL_CLIENT": {"accuracy": 0.483510, "f1_macro": 0.366947, "roc_auc_ovr_macro": 0.635896},
        "TADP_GATE": {"accuracy": 0.505723, "f1_macro": 0.315053, "roc_auc_ovr_macro": 0.636840},
    },
    242: {
        "ALL_CLIENT": {"accuracy": 0.484140, "f1_macro": 0.362154, "roc_auc_ovr_macro": 0.630101},
        "TADP_GATE": {"accuracy": 0.476040, "f1_macro": 0.310190, "roc_auc_ovr_macro": 0.628455},
    },
}


def clean_reference_dataframe() -> pd.DataFrame:
    rows = []
    for seed, by_mode in EXPERIMENT_A_CLEAN_REFERENCE.items():
        for gate_mode, metrics in by_mode.items():
            rows.append({
                "seed": int(seed),
                "gate_mode": str(gate_mode),
                "clean_accuracy": float(metrics["accuracy"]),
                "clean_f1_macro": float(metrics["f1_macro"]),
                "clean_roc_auc_ovr_macro": float(metrics["roc_auc_ovr_macro"]),
                "clean_reference_source": "Experiment A v20.1 completed run",
            })
    return pd.DataFrame(rows)


def _stable_client_seed(base_seed: int, cid: str, salt: int = 0) -> int:
    digest = hashlib.sha256(str(cid).encode("utf-8")).hexdigest()
    return int(base_seed + salt + (int(digest[:8], 16) % 1000000))


def _target_string(class_id: int) -> str:
    mapping = {0: "NO", 1: ">30", 2: "<30"}
    return mapping[int(class_id)]


def copy_documentary_evidence(evidence):
    return copy.deepcopy(evidence)


def derive_backdoor_trigger(
    train_df: pd.DataFrame,
    preprocessor: Any,
    n_features: int = BACKDOOR_TRIGGER_FEATURES,
) -> Dict[str, str]:
    """
    Derive a fixed rare-but-valid categorical conjunction from TRAIN only.

    The trigger uses existing category values, so it does not introduce a new schema
    token. No TEST information is consulted. Preferred low-cardinality clinical fields
    are used when available; otherwise a deterministic categorical fallback is used.
    """
    preferred = [
        "A1Cresult",
        "max_glu_serum",
        "change",
        "diabetesMed",
        "insulin",
        "metformin",
        "glipizide",
        "glyburide",
    ]

    cat_cols = [c for c in preprocessor.categorical_cols if c in train_df.columns]
    ordered = [c for c in preferred if c in cat_cols]

    fallback = []
    for c in cat_cols:
        if c in ordered:
            continue
        s = train_df[c].astype("string").dropna().astype(str)
        nunique = int(s.nunique())
        if 2 <= nunique <= 20:
            fallback.append(c)
    ordered.extend(sorted(fallback))

    trigger = {}
    n_train = len(train_df)
    min_support = max(20, int(round(0.001 * n_train)))

    for c in ordered:
        s = train_df[c].astype("string").dropna().astype(str)
        vc = s.value_counts()
        vc = vc[(vc.index != "__MISSING__") & (vc >= min_support)]
        if len(vc) < 2:
            continue
        # Rare valid value, with deterministic lexical tie-break.
        min_count = int(vc.min())
        candidates = sorted([str(v) for v in vc[vc.eq(min_count)].index.tolist()])
        trigger[c] = candidates[0]
        if len(trigger) >= int(n_features):
            break

    if len(trigger) < int(n_features):
        raise RuntimeError(
            f"Could not derive {n_features} schema-valid categorical trigger fields "
            f"from TRAIN. Derived={trigger}"
        )
    return trigger


def trigger_natural_prevalence(df: pd.DataFrame, trigger: Dict[str, str]) -> float:
    if not trigger:
        return 0.0
    mask = np.ones(len(df), dtype=bool)
    for c, v in trigger.items():
        mask &= df[c].astype("string").fillna("__MISSING__").astype(str).eq(str(v)).to_numpy()
    return float(mask.mean()) if len(mask) else 0.0


def apply_label_flip(
    df: pd.DataFrame,
    fraction: float,
    seed: int,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    out = df.copy(deep=True)
    rng = np.random.default_rng(int(seed))
    n = len(out)
    n_poison = int(max(1, round(float(fraction) * n)))
    idx = rng.choice(n, size=min(n_poison, n), replace=False)
    old_y = out.iloc[idx]["_target"].to_numpy(dtype=np.int32)
    new_y = (old_y + 1) % 3
    out.loc[out.index[idx], "_target"] = new_y
    if TARGET in out.columns:
        out.loc[out.index[idx], TARGET] = [_target_string(y) for y in new_y]
    return out, {
        "records_total": int(n),
        "records_poisoned": int(len(idx)),
        "record_fraction_realized": float(len(idx) / max(1, n)),
        "attack_detail": "cyclic valid-label flip: 0->1, 1->2, 2->0",
    }


def apply_backdoor(
    df: pd.DataFrame,
    trigger: Dict[str, str],
    target_class: int,
    fraction: float,
    seed: int,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    out = df.copy(deep=True)
    rng = np.random.default_rng(int(seed))
    candidate_pos = np.where(out["_target"].to_numpy(dtype=np.int32) != int(target_class))[0]
    if len(candidate_pos) == 0:
        raise RuntimeError("Backdoor attack has no non-target local records to poison.")
    n_target = int(max(1, round(float(fraction) * len(out))))
    n_poison = min(n_target, len(candidate_pos))
    chosen = rng.choice(candidate_pos, size=n_poison, replace=False)
    chosen_index = out.index[chosen]
    for c, v in trigger.items():
        out.loc[chosen_index, c] = str(v)
    out.loc[chosen_index, "_target"] = int(target_class)
    if TARGET in out.columns:
        out.loc[chosen_index, TARGET] = _target_string(target_class)
    return out, {
        "records_total": int(len(out)),
        "records_poisoned": int(n_poison),
        "record_fraction_realized": float(n_poison / max(1, len(out))),
        "attack_detail": json.dumps({
            "target_class": int(target_class),
            "trigger": trigger,
        }, sort_keys=True),
    }


def apply_severe_quality_degradation(
    df: pd.DataFrame,
    preprocessor: Any,
    bad_fraction: float,
    duplicate_fraction: float,
    seed: int,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Fixed severe poor-data stressor used only for PROVENANCE_INFLATION.

    Row count and labels are preserved. A disjoint subset is made feature-duplicate,
    while another subset receives malformed numeric values and extensive missing
    categorical values. The corruption is fixed before governance and never tuned on
    TEST performance.
    """
    out = df.copy(deep=True)
    rng = np.random.default_rng(int(seed))
    n = len(out)
    if n < 5:
        raise RuntimeError("Client shard too small for quality-degradation probe.")

    n_dup = min(int(round(float(duplicate_fraction) * n)), n - 2)
    all_pos = np.arange(n)
    dup_pos = rng.choice(all_pos, size=max(1, n_dup), replace=False)
    remaining = np.setdiff1d(all_pos, dup_pos, assume_unique=False)
    n_bad = min(int(round(float(bad_fraction) * n)), len(remaining))
    bad_pos = rng.choice(remaining, size=max(1, n_bad), replace=False)

    feature_cols = [c for c in preprocessor.feature_cols if c in out.columns]
    donor_candidates = np.setdiff1d(remaining, bad_pos, assume_unique=False)
    donor_pos = int(donor_candidates[0] if len(donor_candidates) else remaining[0])
    donor = out.iloc[donor_pos][feature_cols].copy()

    # Create exact duplicate feature patterns while preserving unique IDs/row IDs.
    for c in feature_cols:
        out.loc[out.index[dup_pos], c] = donor[c]

    # Malformed numeric values: fixed clean preprocessor later coerces them to NaN.
    for c in preprocessor.numeric_cols:
        if c in out.columns:
            out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"

    # Extensive categorical missingness on the same poor-quality rows.
    cat_cols = [c for c in preprocessor.categorical_cols if c in out.columns]
    n_cat_bad = int(max(1, math.ceil(0.70 * len(cat_cols)))) if cat_cols else 0
    for c in cat_cols[:n_cat_bad]:
        out.loc[out.index[bad_pos], c] = np.nan

    return out, {
        "records_total": int(n),
        "records_duplicate_pattern": int(len(dup_pos)),
        "records_malformed_or_missing": int(len(bad_pos)),
        "record_fraction_realized": float((len(dup_pos) + len(bad_pos)) / max(1, n)),
        "attack_detail": (
            f"feature-duplicate fraction={len(dup_pos)/max(1,n):.3f}; "
            f"malformed/missing fraction={len(bad_pos)/max(1,n):.3f}; "
            "labels preserved"
        ),
    }


def inflate_nonmandatory_documentary_scores(
    evidence: Dict[str, Dict[str, Dict[str, float]]],
    malicious_clients: List[str],
    domain: str = DOMAIN,
) -> Dict[str, Dict[str, Dict[str, float]]]:
    """
    Conservative upper-bound provenance inflation stress test.

    Only designated NON-MANDATORY documentary review factors are inflated to 5.
    Mandatory factors are untouched because the stated threat model assumes that a
    third-party artifact that survives verification cannot simply be forged.
    """
    out = copy_documentary_evidence(evidence)
    for cid in malicious_clients:
        for dim, factor_list in REVIEW_FACTORS_BY_DOMAIN[domain].items():
            for factor in factor_list:
                out[cid][dim][factor] = 5.0
    return out


def recompute_dq_and_arrays(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: Any,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, Dict[str, float]], pd.DataFrame, Dict[str, Tuple[np.ndarray, np.ndarray]]]:
    dq_scores = {}
    dq_rows = []
    arrays = {}
    for cid, frame in client_frames.items():
        scores, raw = tabular_dq_scores(frame, preprocessor, reference)
        dq_scores[str(cid)] = scores
        dq_rows.append({"client": str(cid), **scores, **raw})
        arrays[str(cid)] = (
            preprocessor.transform(frame),
            frame["_target"].to_numpy(dtype=np.int32),
        )
    return dq_scores, pd.DataFrame(dq_rows), arrays


def build_triggered_test(
    test_df: pd.DataFrame,
    preprocessor: Any,
    trigger: Dict[str, str],
    target_class: int,
) -> Tuple[np.ndarray, np.ndarray]:
    work = test_df.loc[test_df["_target"].ne(int(target_class))].copy(deep=True)
    if work.empty:
        raise RuntimeError("No non-target TEST rows available for backdoor ASR evaluation.")
    y_true = work["_target"].to_numpy(dtype=np.int32)
    for c, v in trigger.items():
        work[c] = str(v)
    X = preprocessor.transform(work)
    return X, y_true


def evaluate_backdoor_asr(
    model: keras.Model,
    X_triggered: np.ndarray,
    target_class: int,
) -> float:
    p = model.predict(X_triggered, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    return float(np.mean(pred == int(target_class)))


def make_attack_condition(
    clean_clients_raw: Dict[str, pd.DataFrame],
    clean_evidence: Dict[str, Dict[str, Dict[str, float]]],
    malicious_clients: List[str],
    attack: str,
    preprocessor: Any,
    trigger: Dict[str, str],
    condition_seed: int,
) -> Tuple[
    Dict[str, pd.DataFrame],
    Dict[str, Dict[str, Dict[str, float]]],
    pd.DataFrame,
]:
    frames = {cid: df.copy(deep=True) for cid, df in clean_clients_raw.items()}
    evidence = copy_documentary_evidence(clean_evidence)
    audit_rows = []

    for cid in malicious_clients:
        seed = _stable_client_seed(condition_seed, cid, salt=1700)
        if attack == "LABEL_FLIP":
            frames[cid], audit = apply_label_flip(
                frames[cid], LABEL_FLIP_RECORD_FRACTION, seed
            )
        elif attack == "BACKDOOR":
            frames[cid], audit = apply_backdoor(
                frames[cid], trigger, BACKDOOR_TARGET_CLASS,
                BACKDOOR_RECORD_FRACTION, seed,
            )
        elif attack == "PROVENANCE_INFLATION":
            frames[cid], audit = apply_severe_quality_degradation(
                frames[cid], preprocessor,
                PROVENANCE_BAD_RECORD_FRACTION,
                PROVENANCE_DUPLICATE_RECORD_FRACTION,
                seed,
            )
        else:
            raise ValueError(f"Unknown attack={attack!r}")

        audit_rows.append({
            "attack": attack,
            "client": str(cid),
            "malicious": True,
            "condition_seed": int(condition_seed),
            **audit,
        })

    if attack == "PROVENANCE_INFLATION":
        evidence = inflate_nonmandatory_documentary_scores(
            evidence, malicious_clients, domain=DOMAIN
        )

    return frames, evidence, pd.DataFrame(audit_rows)


def build_governance_attack_audit(
    gov: pd.DataFrame,
    malicious_clients: List[str],
    attack: str,
    prevalence: float,
    condition_seed: int,
) -> pd.DataFrame:
    out = gov.copy()
    out["attack"] = str(attack)
    out["prevalence"] = float(prevalence)
    out["condition_seed"] = int(condition_seed)
    out["malicious"] = out["client"].astype(str).isin(set(map(str, malicious_clients)))
    keep = [
        "attack", "prevalence", "condition_seed", "client", "malicious",
        "mandatory_gate_pass", "mandatory_below_minimum", "hps",
        "review_score_0_5", "review_acceptance_threshold_0_5",
        "dimension_floor_failures", "dim1_score_0_5", "dim2_score_0_5",
        "dim3_score_0_5", "dim4_score_0_5", "dim5_score_0_5", "dim6_score_0_5",
        "decision_path", "final_action", "status", "reason",
    ]
    return out[[c for c in keep if c in out.columns]].copy()


def governance_condition_stats(
    gov: pd.DataFrame,
    malicious_clients: List[str],
) -> Dict[str, Any]:
    malicious = set(map(str, malicious_clients))
    g = gov.copy()
    g["client"] = g["client"].astype(str)
    gm = g[g["client"].isin(malicious)]
    gh = g[~g["client"].isin(malicious)]
    admitted_m = gm["final_action"].eq("ACCEPT")
    return {
        "adversary_admitted_count": int(admitted_m.sum()),
        "adversary_total_count": int(len(gm)),
        "adversary_admission_rate": float(admitted_m.mean()) if len(gm) else np.nan,
        "malicious_hps_mean": float(gm["hps"].mean()) if len(gm) else np.nan,
        "honest_hps_mean": float(gh["hps"].mean()) if len(gh) else np.nan,
        "malicious_dq_mean": float(gm["dim2_score_0_5"].mean()) if len(gm) else np.nan,
        "honest_dq_mean": float(gh["dim2_score_0_5"].mean()) if len(gh) else np.nan,
        "mandatory_gate_rejects_malicious": int(
            gm["status"].eq("AUTO_REJECTED_MANDATORY_GATE").sum()
        ),
        "dimension_floor_rejects_malicious": int(
            gm["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()
        ),
        "low_hps_rejects_malicious": int(
            gm["status"].eq("AUTO_REJECTED_LOW_HPS").sum()
        ),
        "review_score_rejects_malicious": int(
            gm["status"].eq("AUTO_REJECTED_REVIEW_SCORE").sum()
        ),
    }


def run_fedavg_condition(
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    build_model_fn,
    initial_weights: List[np.ndarray],
    X_test: np.ndarray,
    y_test: np.ndarray,
    class_weights: Dict[int, float],
    run_seed: int,
    scenario_label: str,
    overall_idx: int,
    overall_total: int,
    train_monitor,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError(f"{scenario_label}: no clients selected for training.")
    selected_per_round = [list(selected_clients) for _ in range(NUM_ROUNDS_FL)]
    return federated_train(
        build_model_fn=build_model_fn,
        initial_weights=initial_weights,
        selected_per_round=selected_per_round,
        client_arrays=client_arrays,
        X_test=X_test,
        y_test=y_test,
        n_classes=3,
        batch_size=BATCH_SIZE,
        local_epochs=LOCAL_EPOCHS,
        class_weights=class_weights,
        run_seed=int(run_seed),
        exact_step_maps=None,
        equal_weight=False,
        fedprox_mu=0.0,
        train_monitor=train_monitor,
        progress_context={
            "scenario": scenario_label,
            "run_idx": 1,
            "run_total": 1,
            "scenario_idx": int(overall_idx),
            "scenario_total": int(overall_total),
            "overall_idx": int(overall_idx),
            "overall_total": int(overall_total),
        },
    )


def append_table(df: pd.DataFrame, path: Path):
    if df is None or df.empty:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, mode="a", header=not path.exists(), index=False)


def _scenario_key(run_idx, attack, prevalence, gate_mode):
    return (
        f"run={int(run_idx)}|attack={attack}|prev={float(prevalence):.2f}|"
        f"gate={gate_mode}"
    )


def summarize_adversarial_results(perf: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    p = perf.copy()
    clean = clean_reference_dataframe()
    attacked = p.merge(
        clean,
        on=["seed", "gate_mode"],
        how="left",
        validate="many_to_one",
    )
    if attacked[["clean_accuracy", "clean_f1_macro", "clean_roc_auc_ovr_macro"]].isna().any().any():
        raise RuntimeError("Missing Experiment A clean reference for one or more training rows.")

    attacked["accuracy_degradation"] = attacked["clean_accuracy"] - attacked["accuracy"]
    attacked["f1_degradation"] = attacked["clean_f1_macro"] - attacked["f1_macro"]
    attacked["auc_degradation"] = attacked["clean_roc_auc_ovr_macro"] - attacked["roc_auc_ovr_macro"]

    metrics = [
        "adversary_admission_rate", "malicious_hps_mean", "honest_hps_mean",
        "malicious_dq_mean", "honest_dq_mean", "accuracy", "f1_macro",
        "roc_auc_ovr_macro", "accuracy_degradation", "f1_degradation",
        "auc_degradation", "backdoor_asr", "selected_client_count",
        "optimizer_steps", "runtime_s", "communication_mb",
    ]
    rows = []
    for keys, d in attacked.groupby(["attack", "prevalence", "gate_mode"], sort=False):
        attack, prev, gate_mode = keys
        row = {
            "attack": attack,
            "prevalence": float(prev),
            "gate_mode": gate_mode,
            "n_runs": int(len(d)),
        }
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return attacked, pd.DataFrame(rows)

def write_adversarial_figures(
    summary: pd.DataFrame,
    governance_summary: pd.DataFrame,
    out_dir: Path,
):
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"Figure generation skipped: {exc}")
        return

    # Figure 1: governance-only admission rate across 10%, 20%, and 30%.
    d = governance_summary.copy()
    if not d.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        for attack, g in d.groupby("attack", sort=False):
            g = g.sort_values("prevalence")
            ax.plot(
                100.0 * g["prevalence"].to_numpy(dtype=float),
                100.0 * g["adversary_admission_rate"].to_numpy(dtype=float),
                marker="o",
                label=attack.replace("_", " ").title(),
            )
        ax.set_xlabel("Malicious clients (% of K=10)")
        ax.set_ylabel("Adversarial client admission rate (%)")
        ax.set_ylim(-5, 105)
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(Path(out_dir) / "Figure_Adversarial_Admission_Rate_GovernanceOnly.png", dpi=220)
        plt.close(fig)

    # Figure 2: macro-F1 degradation at the single trained contamination level (20%).
    if not summary.empty:
        work = summary.copy()
        labels = []
        values = []
        for _, r in work.iterrows():
            labels.append(
                f"{str(r['attack']).replace('_',' ').title()}\n{str(r['gate_mode']).replace('_',' ').title()}"
            )
            values.append(float(r["f1_degradation_mean"]))
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(np.arange(len(values)), values)
        ax.axhline(0.0, linewidth=1.0)
        ax.set_xticks(np.arange(len(values)))
        ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.set_ylabel("Macro-F1 degradation vs Experiment A clean reference")
        ax.set_title("Adversarial diagnostic at 20% malicious clients")
        fig.tight_layout()
        fig.savefig(Path(out_dir) / "Figure_Adversarial_F1_Degradation_20pct.png", dpi=220)
        plt.close(fig)

    # Figure 3: absolute backdoor ASR at 20%.
    b = summary[summary["attack"].eq("BACKDOOR")].copy()
    if not b.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        labels = b["gate_mode"].str.replace("_", " ", regex=False).str.title().tolist()
        vals = 100.0 * b["backdoor_asr_mean"].to_numpy(dtype=float)
        ax.bar(np.arange(len(vals)), vals)
        ax.set_xticks(np.arange(len(vals)))
        ax.set_xticklabels(labels)
        ax.set_ylabel("Backdoor attack success rate (%)")
        ax.set_ylim(0, 100)
        ax.set_title("Backdoor diagnostic at 20% malicious clients")
        fig.tight_layout()
        fig.savefig(Path(out_dir) / "Figure_Backdoor_ASR_20pct.png", dpi=220)
        plt.close(fig)

def main():
    csv_path = locate_diabetes_csv()

    print_banner(EXPERIMENT_VERSION)
    print("Purpose: diagnostic adversarial probe; NOT a poisoning/backdoor defense evaluation.")
    print(f"Training seeds: {TRAINING_RUN_SEEDS}")
    print(f"Clients: K={NUM_CLIENTS}")
    print(f"Governance-only contamination levels: {CONTAMINATION_LEVELS} -> 1, 2, 3 malicious clients")
    print(f"Model-training contamination level: {TRAINING_CONTAMINATION_LEVEL:.0%} -> 2 malicious clients")
    print(f"Attacks: {ATTACKS}")
    print(f"FL rounds: {NUM_ROUNDS_FL}; local epochs: {LOCAL_EPOCHS}; batch: {BATCH_SIZE}")
    print("Malicious clients are selected only from the clean TADP-VR eligible cohort.")
    print("Mandatory evidence remains genuine; provenance inflation affects non-mandatory documentary factors only.")

    data = prepare_diabetes_no_leakage(
        csv_path=csv_path,
        split_seed=GLOBAL_SPLIT_SEED,
        partition_seed=CLIENT_PARTITION_SEED,
        n_clients=NUM_CLIENTS,
        alpha=DIRICHLET_ALPHA,
    )

    leakage = write_leakage_audit(
        EXPERIMENT_ROOT,
        data["global_train_ids"],
        data["global_test_ids"],
        data["client_train_ids"],
        extra={
            "patient_overlap": int(data["meta"]["patient_overlap"]),
            "global_holdout_before_client_partition": True,
            "preprocessing_train_only": True,
            "dq_ge_tadp_train_only": True,
            "test_used_for_final_evaluation_only": True,
            "trigger_derived_from_train_only": True,
        },
    )
    print_banner("NO-LEAKAGE AUDIT")
    print(pd.DataFrame([leakage]).to_string(index=False))

    client_ids = list(map(str, data["client_ids"]))
    clean_evidence, evidence_df = generate_controlled_documentary_evidence(
        client_ids,
        FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        DOMAIN_FACTOR_MINIMA[DOMAIN],
        domain=DOMAIN,
    )
    evidence_df.to_csv(EXPERIMENT_ROOT / "clean_controlled_documentary_evidence.csv", index=False)

    clean_gov = build_tadp_governance(
        client_ids,
        clean_evidence,
        data["dq_scores"],
        run=0,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain=DOMAIN,
    )
    clean_gov.to_csv(EXPERIMENT_ROOT / "clean_governance.csv", index=False)
    clean_vr = accepted_tadp_vr(clean_gov)
    if len(clean_vr) < 3:
        raise RuntimeError(
            f"Need at least three clean TADP-eligible clients for 30% attack prevalence; got {clean_vr}"
        )

    print_banner("CLEAN TADP GOVERNANCE")
    cols = [
        "client", "mandatory_gate_pass", "hps", "review_score_0_5",
        "dim2_score_0_5", "final_action", "status", "reason",
    ]
    print(clean_gov[[c for c in cols if c in clean_gov.columns]].to_string(index=False))
    print(f"Clean TADP-VR eligible cohort: {len(clean_vr)}/{NUM_CLIENTS} -> {clean_vr}")

    rng_attackers = np.random.default_rng(int(ATTACK_CLIENT_SELECTION_SEED))
    eligible_order = list(rng_attackers.permutation(clean_vr))
    attacker_sets = {}
    for prevalence in CONTAMINATION_LEVELS:
        n_malicious = int(round(float(prevalence) * NUM_CLIENTS))
        if n_malicious < 1:
            n_malicious = 1
        attacker_sets[float(prevalence)] = list(map(str, eligible_order[:n_malicious]))

    print_banner("FROZEN MALICIOUS-CLIENT ASSIGNMENT")
    for prev, ids in attacker_sets.items():
        print(f"{prev:.0%}: {ids}")

    trigger = derive_backdoor_trigger(
        data["train_df"], data["preprocessor"], BACKDOOR_TRIGGER_FEATURES
    )
    natural_trigger_train = trigger_natural_prevalence(data["train_df"], trigger)
    natural_trigger_test = trigger_natural_prevalence(data["test_df"], trigger)
    X_triggered_test, y_triggered_true = build_triggered_test(
        data["test_df"], data["preprocessor"], trigger, BACKDOOR_TARGET_CLASS
    )
    print_banner("BACKDOOR TRIGGER — DERIVED FROM TRAIN ONLY")
    print(json.dumps(trigger, indent=2, sort_keys=True))
    print(f"Natural conjunction prevalence in TRAIN: {natural_trigger_train:.6f}")
    print(f"Natural conjunction prevalence in held-out TEST before trigger insertion: {natural_trigger_test:.6f}")
    print(f"Triggered TEST evaluation rows (non-target originals): {len(y_triggered_true)}")

    design = {
        "experiment_version": EXPERIMENT_VERSION,
        "purpose": "diagnostic adversarial probe, not defense evaluation",
        "domain": DOMAIN,
        "training_seeds": TRAINING_RUN_SEEDS,
        "K": NUM_CLIENTS,
        "rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "governance_only_contamination_levels": CONTAMINATION_LEVELS,
        "training_contamination_level": TRAINING_CONTAMINATION_LEVEL,
        "attacker_sets": {str(k): v for k, v in attacker_sets.items()},
        "clean_tadp_vr": clean_vr,
        "label_flip_record_fraction": LABEL_FLIP_RECORD_FRACTION,
        "backdoor_record_fraction": BACKDOOR_RECORD_FRACTION,
        "backdoor_target_class": BACKDOOR_TARGET_CLASS,
        "backdoor_trigger": trigger,
        "backdoor_trigger_natural_train_prevalence": natural_trigger_train,
        "backdoor_trigger_natural_test_prevalence": natural_trigger_test,
        "provenance_bad_record_fraction": PROVENANCE_BAD_RECORD_FRACTION,
        "provenance_duplicate_record_fraction": PROVENANCE_DUPLICATE_RECORD_FRACTION,
        "provenance_inflation_scope": "non-mandatory documentary review factors only; mandatory factors unchanged",
        "mandatory_gate": MANDATORY_FACTORS_BY_DOMAIN[DOMAIN],
        "review_factors": REVIEW_FACTORS_BY_DOMAIN[DOMAIN],
        "dimension_floor": DIMENSION_MIN_FLOOR,
        "hps_reject_threshold": GOOD_CUT,
        "hps_accept_threshold": HIGH_CUT,
        "review_acceptance_threshold": REVIEW_ACCEPT_THRESHOLD,
        "clean_reference_source": "completed Experiment A v20.1; exact per-seed held-out TEST metrics reused, no clean retraining",
        "backdoor_asr_reference": "absolute ASR only; clean Experiment A models were not retained for triggered-test evaluation",
        "test_semantics": "clean TEST and triggered TEST copies used only after training for final diagnostic evaluation",
        "excluded_threats": [
            "arbitrary Byzantine/model-replacement updates",
            "server compromise",
            "ledger compromise",
            "collusion",
            "cryptographic forgery",
        ],
    }
    atomic_write_json(design, EXPERIMENT_ROOT / "threat_model_and_experiment_design.json")
    clean_reference_dataframe().to_csv(EXPERIMENT_ROOT / "experiment_A_v20_1_clean_reference_metrics.csv", index=False)

    # Precompute all attack conditions once. Governance and client selection are
    # attack-condition properties and must remain frozen across training seeds.
    conditions = {}
    all_gov_audits = []
    all_attack_audits = []
    for attack_idx, attack in enumerate(ATTACKS):
        for prevalence in CONTAMINATION_LEVELS:
            malicious_clients = attacker_sets[float(prevalence)]
            condition_seed = int(
                ATTACK_CLIENT_SELECTION_SEED
                + 10000 * (attack_idx + 1)
                + int(round(100 * prevalence))
            )
            frames, evidence, attack_audit = make_attack_condition(
                data["clients_raw"],
                clean_evidence,
                malicious_clients,
                attack,
                data["preprocessor"],
                trigger,
                condition_seed,
            )
            dq_scores, dq_audit, arrays = recompute_dq_and_arrays(
                frames,
                data["preprocessor"],
                data["dq_reference"],
            )
            gov = build_tadp_governance(
                client_ids,
                evidence,
                dq_scores,
                run=0,
                evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
                domain=DOMAIN,
            )
            accepted = accepted_tadp_vr(gov)
            gov_stats = governance_condition_stats(gov, malicious_clients)

            # This is a diagnostic, not an expected-outcome assertion. We record
            # whatever the policy actually does for each attack condition.
            key = (attack, float(prevalence))
            conditions[key] = {
                "frames": frames,
                "evidence": evidence,
                "dq_scores": dq_scores,
                "dq_audit": dq_audit,
                "arrays": arrays,
                "gov": gov,
                "accepted": accepted,
                "malicious_clients": malicious_clients,
                "condition_seed": condition_seed,
                "gov_stats": gov_stats,
            }
            if not attack_audit.empty:
                attack_audit = attack_audit.copy()
                attack_audit["prevalence"] = float(prevalence)
                attack_audit["malicious_clients"] = ";".join(malicious_clients)
                all_attack_audits.append(attack_audit)
            all_gov_audits.append(
                build_governance_attack_audit(
                    gov, malicious_clients, attack, prevalence, condition_seed
                )
            )

            print_banner(f"GOVERNANCE PRECHECK | {attack} | prevalence={prevalence:.0%}")
            print(f"Malicious clients: {malicious_clients}")
            print(f"TADP admitted cohort: {len(accepted)}/{NUM_CLIENTS} -> {accepted}")
            print(
                f"Adversarial admission rate: {gov_stats['adversary_admission_rate']:.3f} | "
                f"malicious mean HPS={gov_stats['malicious_hps_mean']:.3f} | "
                f"malicious mean DQ={gov_stats['malicious_dq_mean']:.3f}"
            )
            show = gov[gov["client"].isin(malicious_clients)][[
                "client", "mandatory_gate_pass", "hps", "review_score_0_5",
                "dim2_score_0_5", "dimension_floor_failures", "final_action",
                "status", "reason",
            ]]
            print(show.to_string(index=False))

    pd.concat(all_gov_audits, ignore_index=True).to_csv(GOV_AUDIT_PATH, index=False)
    if all_attack_audits:
        pd.concat(all_attack_audits, ignore_index=True).to_csv(ATTACK_AUDIT_PATH, index=False)

    # Model training is intentionally restricted to the 20% contamination level.
    # Governance-only diagnostics above still cover 10%, 20%, and 30%.
    training_prevalence = float(TRAINING_CONTAMINATION_LEVEL)
    n_attack_conditions = len(ATTACKS) * len(GATE_MODES)
    total_training_exec = len(TRAINING_RUN_SEEDS) * n_attack_conditions
    print_banner("TRAINING PLAN — COMPACT REVIEWER-RESPONSE DESIGN")
    print("Clean models are NOT retrained; exact per-seed clean metrics are reused from Experiment A v20.1.")
    print(f"Training prevalence: {training_prevalence:.0%}")
    print(f"Attacked scenarios per seed: {n_attack_conditions} = 3 attacks x 2 modes")
    print(f"Training seeds: {len(TRAINING_RUN_SEEDS)}")
    print(f"Total model-training executions: {total_training_exec}")

    clean_ref_table = clean_reference_dataframe()
    clean_ref_table.to_csv(EXPERIMENT_ROOT / "experiment_A_v20_1_embedded_clean_references.csv", index=False)
    print_banner("FROZEN CLEAN REFERENCES — COMPLETED EXPERIMENT A v20.1")
    print(clean_ref_table.to_string(index=False))
    print(
        "These reference metrics are used only after each attacked run to calculate "
        "seed-matched degradation; they do not affect attack construction, governance, "
        "client selection, preprocessing, or training."
    )

    state = load_checkpoint_state(CHECKPOINT_STATE)
    completed = set(state.get("completed", []))
    overall_counter = 0

    for run_idx, seed in enumerate(TRAINING_RUN_SEEDS, start=1):
        seed_everything(seed)
        build_model_fn = lambda: build_diabetes_model(data["meta"]["input_dim"], LEARNING_RATE)
        w0_model = build_model_fn()
        initial_weights = [np.array(w, copy=True) for w in w0_model.get_weights()]
        w0_hash = sha256_weights(initial_weights)
        del w0_model
        tf.keras.backend.clear_session()

        for attack in ATTACKS:
            prevalence = training_prevalence
            cond = conditions[(attack, prevalence)]
            attacked_arrays = cond["arrays"]
            train_monitor_attacked = build_train_monitor_subset(
                attacked_arrays,
                seed=99117 + int(100 * prevalence) + 1000 * ATTACKS.index(attack),
            )

            for gate_mode in GATE_MODES:
                overall_counter += 1
                key = _scenario_key(run_idx, attack, prevalence, gate_mode)
                if key in completed:
                    print(f"SKIP checkpoint: {key}")
                    continue

                selected = client_ids if gate_mode == "ALL_CLIENT" else list(cond["accepted"])
                if not selected:
                    raise RuntimeError(
                        f"No clients remain for training under {attack} / {gate_mode}. "
                        "Record this as a governance outcome rather than silently training an empty set."
                    )

                scenario = f"{attack} | prevalence={prevalence:.0%} | {gate_mode}"
                result = run_fedavg_condition(
                    selected,
                    attacked_arrays,
                    build_model_fn,
                    initial_weights,
                    data["X_test"],
                    data["y_test"],
                    data["class_weights"],  # frozen clean TRAIN-only class weights
                    seed,
                    scenario,
                    overall_counter,
                    total_training_exec,
                    train_monitor_attacked,
                )
                asr = evaluate_backdoor_asr(
                    result["model"], X_triggered_test, BACKDOOR_TARGET_CLASS
                )
                row = result_row(run_idx, seed, scenario, result, w0_hash)

                # Immediate seed-matched comparison against the already completed
                # clean Experiment A v20.1 result. This is reporting only.
                ref = EXPERIMENT_A_CLEAN_REFERENCE[int(seed)][str(gate_mode)]
                clean_acc = float(ref["accuracy"])
                clean_f1 = float(ref["f1_macro"])
                clean_auc = float(ref["roc_auc_ovr_macro"])
                attacked_acc = float(row["accuracy"])
                attacked_f1 = float(row["f1_macro"])
                attacked_auc = float(row["roc_auc_ovr_macro"])

                print_banner(
                    f"SEED-MATCHED CLEAN vs ATTACKED | seed={seed} | "
                    f"{attack} | {gate_mode}"
                )
                print(
                    f"Accuracy : clean={clean_acc:.6f} | attacked={attacked_acc:.6f} | "
                    f"degradation={clean_acc-attacked_acc:+.6f}"
                )
                print(
                    f"Macro-F1 : clean={clean_f1:.6f} | attacked={attacked_f1:.6f} | "
                    f"degradation={clean_f1-attacked_f1:+.6f}"
                )
                print(
                    f"Macro-AUC: clean={clean_auc:.6f} | attacked={attacked_auc:.6f} | "
                    f"degradation={clean_auc-attacked_auc:+.6f}"
                )

                stats = dict(cond["gov_stats"])
                row.update({
                    "attack": attack,
                    "prevalence": float(prevalence),
                    "gate_mode": gate_mode,
                    "malicious_clients": ";".join(cond["malicious_clients"]),
                    **stats,
                    "selected_client_count": int(len(selected)),
                    "selected_clients": ";".join(selected),
                    "backdoor_asr": float(asr),
                    "backdoor_target_class": int(BACKDOOR_TARGET_CLASS),
                    "condition_seed": int(cond["condition_seed"]),
                    "initial_weights_sha256": w0_hash,
                    "clean_reference_reused": True,
                    "clean_reference_experiment": "TADP-A-v20.1",
                })
                upsert_csv(row, PERF_CHECKPOINT, ["run", "attack", "prevalence", "gate_mode"])
                mark_checkpoint_complete(CHECKPOINT_STATE, key)
                completed.add(key)
                del result["model"]
                gc.collect()
                tf.keras.backend.clear_session()

    perf = pd.read_csv(PERF_CHECKPOINT)
    attacked_detail, summary = summarize_adversarial_results(perf)
    perf.to_csv(EXPERIMENT_ROOT / "adversarial_performance_all_runs.csv", index=False)
    attacked_detail.to_csv(EXPERIMENT_ROOT / "adversarial_attacked_runs_with_degradation.csv", index=False)
    summary.to_csv(EXPERIMENT_ROOT / "adversarial_summary_mean_sd.csv", index=False)

    # Governance-only concise summary (same across training seeds).
    gsum_rows = []
    for (attack, prevalence), cond in conditions.items():
        gsum_rows.append({
            "attack": attack,
            "prevalence": float(prevalence),
            "malicious_clients": ";".join(cond["malicious_clients"]),
            "tadp_admitted_clients": ";".join(cond["accepted"]),
            "tadp_admitted_count": int(len(cond["accepted"])),
            **cond["gov_stats"],
        })
    gov_summary = pd.DataFrame(gsum_rows)
    gov_summary.to_csv(EXPERIMENT_ROOT / "adversarial_governance_summary.csv", index=False)

    write_adversarial_figures(summary, gov_summary, EXPERIMENT_ROOT)

    print_banner("ADVERSARIAL DIAGNOSTIC — GOVERNANCE SUMMARY")
    print(gov_summary.to_string(index=False))
    print_banner("ADVERSARIAL DIAGNOSTIC — TRAINING SUMMARY")
    summary_cols = [
        "attack", "prevalence", "gate_mode", "n_runs",
        "adversary_admission_rate_mean", "f1_macro_mean", "f1_degradation_mean",
        "roc_auc_ovr_macro_mean", "auc_degradation_mean", "backdoor_asr_mean",
        "selected_client_count_mean",
    ]
    print(summary[[c for c in summary_cols if c in summary.columns]].to_string(index=False))

    print_banner("INTERPRETATION GUARDRAILS")
    print("1) This experiment diagnoses what the current TADP gate does under three data/evidence attacks.")
    print("2) It is NOT evidence that TADP is a general poisoning/backdoor/Byzantine defense.")
    print("3) Governance admission is diagnosed at 10%, 20%, and 30%; model effects are trained only at 20% contamination.")
    print("4) Label-flip and backdoor attackers may remain admissible when provenance and measured DQ remain acceptable.")
    print("5) Provenance inflation is limited only to the extent that mandatory evidence is verified and poor DQ is machine-measured.")
    print("6) Absolute backdoor ASR is reported; ASR lift is not claimed because the clean Experiment A models were not retained for triggered-test evaluation.")
    print("7) Arbitrary malicious model updates, collusion, hardened ledger/storage, and cryptographic runtime protection remain TADP-Sec scope.")

    return EXPERIMENT_ROOT


if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(finished_root, EXPERIMENT_VERSION)


Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_G_TADP-G-v20.2-ADVERSARIAL-DIAGNOSTIC-K10-3SEED-20PCT-18RUN-MANDATORYGATE-REVIEWSCORE325-EMBEDDED-CLEAN-REF

TADP-G-v20.2-ADVERSARIAL-DIAGNOSTIC-K10-3SEED-20PCT-18RUN-MANDATORYGATE-REVIEWSCORE325-EMBEDDED-CLEAN-REF
Purpose: diagnostic adversarial probe; NOT a poisoning/backdoor defense evaluation.
Training seeds: [42, 142, 242]
Clients: K=10
Governance-only contamination levels: [0.1, 0.2, 0.3] -> 1, 2, 3 malicious clients
Model-training contamination level: 20% -> 2 malicious clients
Attacks: ['LABEL_FLIP', 'BACKDOOR', 'PROVENANCE_INFLATION']
FL rounds: 4; local epochs: 1; batch: 64
Malicious clients are selected only from the clean TADP-VR eligible cohort.
Mandatory evidence remains genuine; provenance inflation affects non-mandatory documentary factors only.

NO-LEAKAGE AUDIT
 train_test_overlap  test_rows_in_any_client  train_rows_missing_from_cl

/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__


GOVERNANCE PRECHECK | PROVENANCE_INFLATION | prevalence=10%
Malicious clients: ['J']
TADP admitted cohort: 5/10 -> ['A', 'B', 'G', 'H', 'I']
Adversarial admission rate: 0.000 | malicious mean HPS=4.211 | malicious mean DQ=1.125
client  mandatory_gate_pass     hps  review_score_0_5  dim2_score_0_5 dimension_floor_failures final_action                        status                                                     reason
     J                 True 4.21125               5.0           1.125                     dim2       REJECT AUTO_REJECTED_DIMENSION_FLOOR At least one averaged dimension is below 2.5/5: dim2=1.125


/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__


GOVERNANCE PRECHECK | PROVENANCE_INFLATION | prevalence=20%
Malicious clients: ['J', 'H']
TADP admitted cohort: 4/10 -> ['A', 'B', 'G', 'I']
Adversarial admission rate: 0.000 | malicious mean HPS=4.057 | malicious mean DQ=1.188
client  mandatory_gate_pass     hps  review_score_0_5  dim2_score_0_5 dimension_floor_failures final_action                        status                                                     reason
     H                 True 3.88375               5.0           1.125                     dim2       REJECT AUTO_REJECTED_DIMENSION_FLOOR At least one averaged dimension is below 2.5/5: dim2=1.125
     J                 True 4.23000               5.0           1.250                     dim2       REJECT AUTO_REJECTED_DIMENSION_FLOOR At least one averaged dimension is below 2.5/5: dim2=1.250


/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__"
/tmp/ipykernel_2127/3534033336.py:4492: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '__INVALID_NUMERIC__' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  out.loc[out.index[bad_pos], c] = "__INVALID_NUMERIC__


GOVERNANCE PRECHECK | PROVENANCE_INFLATION | prevalence=30%
Malicious clients: ['J', 'H', 'G']
TADP admitted cohort: 3/10 -> ['A', 'B', 'I']
Adversarial admission rate: 0.000 | malicious mean HPS=4.067 | malicious mean DQ=1.167
client  mandatory_gate_pass     hps  review_score_0_5  dim2_score_0_5 dimension_floor_failures final_action                        status                                                     reason
     G                 True 4.10750               5.0           1.250                     dim2       REJECT AUTO_REJECTED_DIMENSION_FLOOR At least one averaged dimension is below 2.5/5: dim2=1.250
     H                 True 3.88375               5.0           1.125                     dim2       REJECT AUTO_REJECTED_DIMENSION_FLOOR At least one averaged dimension is below 2.5/5: dim2=1.125
     J                 True 4.21125               5.0           1.125                     dim2       REJECT AUTO_REJECTED_DIMENSION_FLOOR At least one averaged dimension is below 2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>